# 第4日 ノートブック02：YOLO モデルを学習する

# Day 4 Notebook 02: Training a YOLO Model

<!-- ne:begin after-heading -->
# दिन ४ नोटबुक ०२: YOLO model प्रशिक्षण
<!-- ne:end -->

---

## 学習目標 / Learning objectives

| 日本語 | English |
| --- | --- |
| 1. 学習の流れ（epoch・batch・loss）を説明できる | 1. Explain the training loop (epoch, batch, loss) |
| 2. ファインチューニングの意味を説明できる | 2. Explain what fine-tuning is |
| 3. CPU で現実的な時間で学習を実行できる | 3. Run a CPU-feasible training job |
| 4. 学習の進み具合を読み取れる | 4. Read training progress |
| 5. 学習結果（重み・設定・指標・ログ）を保存できる | 5. Save weights, config, metrics and logs |

## このノートブックの基本情報 / Notebook facts

| 項目 / Item | 内容 / Details |
| --- | --- |
| 学習時間（目安） / Estimated duration | 115 分 / 115 minutes |
| 難易度 / Difficulty | 初級 / Beginner |
| 前提条件 / Prerequisites | 第4日ノートブック01 を終え、`assets/yolo_dataset` が生成されていること。 / You finished Day 4 Notebook 01 and `assets/yolo_dataset` exists. |
| 必要なライブラリ / Required libraries | torch, ultralytics, opencv-python, numpy, matplotlib, pandas |
| 使用カーネル / Jupyter kernel | aivision |
| この日の時間割 / Today's timetable | [`COURSE_SYLLABUS.md` 第4日 / Day 4](../../COURSE_SYLLABUS.md#day4) |

### 提出物 / Expected deliverables

| 日本語 | English |
| --- | --- |
| 学習済み重みファイル | The trained weight file |
| 学習ログと指標 | Training logs and metrics |
| 学習設定の記録 | A record of the training configuration |

<!-- ne:begin end -->
## सिकाइ लक्ष्यहरू

| नेपाली |
| --- |
| 1. प्रशिक्षणको क्रम (epoch, batch, loss) बताउन सक्ने। |
| 2. fine-tuning को अर्थ बताउन सक्ने। |
| 3. CPU मा व्यावहारिक समयभित्र प्रशिक्षण चलाउन सक्ने। |
| 4. प्रशिक्षणको प्रगति पढ्न सक्ने। |
| 5. प्रशिक्षणको नतिजा (weight, सेटिङ, मापन, लग) सुरक्षित गर्न सक्ने। |

## यो नोटबुकको आधारभूत जानकारी

| विषय | विवरण |
| --- | --- |
| सिकाइ समय (अनुमान) | 115 मिनेट |
| कठिनाइ | सुरुवाती तह |
| पूर्वशर्त | दिन ४ नोटबुक ०१ सकिएको र `assets/yolo_dataset` बनिसकेको हुनुपर्छ। |
| आवश्यक लाइब्रेरी | torch, ultralytics, opencv-python, numpy, matplotlib, pandas |
| प्रयोग हुने kernel | aivision |
| आजको समयतालिका | [`COURSE_SYLLABUS.md` दिन ४](../../COURSE_SYLLABUS.md#day4) |

### बुझाउनुपर्ने काम

| नेपाली |
| --- |
| प्रशिक्षण भएको weight फाइल |
| प्रशिक्षणको लग र मापन |
| प्रशिक्षण सेटिङको अभिलेख |
<!-- ne:end -->


## このノートブックの位置 / Where this notebook sits

<!-- ne:begin after-heading -->
## यो नोटबुक कहाँ पर्छ
<!-- ne:end -->

| 項目 / Item | 内容 / Details |
| --- | --- |
| 日 / Day | 第4日 / Day 4 |
| この日の順番 / Position today | 2 / 3（この日のノートブックのうち）/ of today's notebooks |
| 所要時間 / Duration | 115 分 / 115 minutes |
| 前提 / Prerequisites | `Day04_01_YOLO_Dataset_Annotation.ipynb`、`assets/yolo_dataset` 生成済み |
| この日の時間割 / Today's timetable | [`COURSE_SYLLABUS.md` 第4日 / Day 4](../../COURSE_SYLLABUS.md#day4) |

1 日の時間割は `COURSE_SYLLABUS.md` に 1 つだけあります。このノートブックはその一部で、
自分の時間割を持ちません。時刻が分からなくなったら、まずそこを見てください。

The day's timetable exists once, in `COURSE_SYLLABUS.md`. This notebook is one part of it and
does not keep its own schedule; look there first when you are unsure of the time.

<!-- ne:begin end -->
| विषय | विवरण |
| --- | --- |
| दिन | दिन ४ |
| आजको क्रम | आजको नोटबुकमध्ये 2 / 3 |
| लाग्ने समय | 115 मिनेट |
| पूर्वशर्त | `Day04_01_YOLO_Dataset_Annotation.ipynb`, `assets/yolo_dataset` बनिसकेको |
| आजको समयतालिका | [`COURSE_SYLLABUS.md` दिन ४](../../COURSE_SYLLABUS.md#day4) |

दिनको समयतालिका `COURSE_SYLLABUS.md` मा एउटै ठाउँमा छ। यो नोटबुक त्यसको एउटा भाग मात्र हो।
<!-- ne:end -->


### 予想して、実行して、説明する / Predict, run, inspect, explain

<!-- ne:begin after-heading -->
### अनुमान गर्नुहोस्, चलाउनुहोस्, हेर्नुहोस्, व्याख्या गर्नुहोस्
<!-- ne:end -->

epoch はデータを1周、batch は一度に渡す画像群です。更新回数とは区別します。 重要なセルの前に、入力・操作・出力を一文ずつ説明してください。
予想を書いて実行し、値または図を観察します。batch だけ（予測してから実行）を変え、もう一度予想して実行します。
意図的なエラーと未完成の演習は、その指示に従って取り組みます。

An epoch is a pass through data; a batch is a group processed together. Distinguish optimizer updates. Before an important cell, describe its input, operation and output in one sentence each.
Predict, run, inspect the values/image and explain why. Change only batch (predict before running), predict and rerun.
Follow the instructions for deliberately erroneous cells and unsolved exercises.

<!-- ne:begin end -->
epoch भनेको डेटा एक फेर घुम्नु हो, batch भनेको एकै पटक दिइने तस्बिरको समूह हो। यसलाई weight अपडेट
हुने सङ्ख्यासँग नमिसाउनुहोस्। महत्त्वपूर्ण सेल चलाउनुअघि इनपुट, काम र नतिजा एक-एक वाक्यमा लेख्नुहोस्।
पहिले अनुमान लेखेर चलाउनुहोस्, अनि `batch` मात्र बदलेर फेरि अनुमान गरी चलाउनुहोस्।
<!-- ne:end -->


---

## Part 0　準備 / Preparation

<!-- ne:begin after-heading -->
## Part 0 तयारी
<!-- ne:end -->


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd().resolve()
while not ((COURSE_ROOT / "assets").is_dir() and (COURSE_ROOT / "notebooks").is_dir()):
    if COURSE_ROOT.parent == COURSE_ROOT:
        raise FileNotFoundError(
            "研修フォルダが見つかりません。JupyterLab を AI_Vision_Training で起動してください。\n"
            "Course folder not found. Start JupyterLab inside the AI_Vision_Training folder."
        )
    COURSE_ROOT = COURSE_ROOT.parent

ASSETS = COURSE_ROOT / "assets"
sys.path.insert(0, str(ASSETS / "helpers"))
import aivision

print("研修フォルダ / course root :", COURSE_ROOT)
print("素材フォルダ / assets      :", ASSETS)

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
import importlib
import sys

print("Python version:", sys.version.split()[0])

REQUIRED = [
    ("numpy", "numpy"),
    ("matplotlib", "matplotlib"),
    ("cv2", "opencv-python"),
    ("torch", "torch"),
    ("ultralytics", "ultralytics"),
    ("pandas", "pandas"),
]

missing = []
for import_name, pip_name in REQUIRED:
    try:
        module = importlib.import_module(import_name)
        version = getattr(module, "__version__", "unknown")
        print(f"  [OK]      {pip_name:<20} {version}")
    except ImportError:
        print(f"  [MISSING] {pip_name:<20} -> pip install {pip_name}")
        missing.append(pip_name)

if missing:
    print("\n未導入のパッケージがあります / Some packages are missing:", ", ".join(missing))
    print("docs/troubleshooting/installation.md を確認してください / see that file for help.")
else:
    print("\nこのノートブックに必要なライブラリはすべて揃っています。")
    print("All libraries required by this notebook are available.")

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
FONT_FAMILY = aivision.configure_japanese_font()
aivision.set_plot_style()

if FONT_FAMILY is None:
    print("日本語フォントが見つかりません。図のタイトルは英字になります。")
    print("No Japanese font found; figure titles fall back to English.")
else:
    print("図の日本語フォント / figure font:", FONT_FAMILY)

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
import cv2
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

def show(images, titles=None, figsize=(14, 5)):
    """画像を並べて表示する / display images side by side."""
    if not isinstance(images, (list, tuple)):
        images = [images]
    if titles is None:
        titles = [""] * len(images)
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    if len(images) == 1:
        axes = [axes]
    for ax, image, title in zip(axes, images, titles):
        ax.imshow(image)
        ax.set_title(title, fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()


def detections_to_frame(result):
    """検出結果を pandas の表にする / convert detections into a DataFrame.

    ``result.boxes.data`` の 6 列は
    (x1, y1, x2, y2, confidence, class_id) です。
    """
    columns = ["class_id", "name", "confidence", "x1", "y1", "x2", "y2"]
    boxes = result.boxes
    if len(boxes) == 0:
        return pd.DataFrame(columns=columns)

    data = boxes.data.cpu().numpy()
    return pd.DataFrame({
        "class_id": data[:, 5].astype(int),
        "name": [result.names[int(class_id)] for class_id in data[:, 5]],
        "confidence": data[:, 4].round(4),
        "x1": data[:, 0].round(2),
        "y1": data[:, 1].round(2),
        "x2": data[:, 2].round(2),
        "y2": data[:, 3].round(2),
    })

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
import shutil
from pathlib import Path

MODELS_DIR = ASSETS / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

def load_pretrained_yolo(model_name="yolo26n.pt"):
    """学習済みモデルを読み込む。初回だけダウンロードする。
    Load a pretrained model, downloading it only on the first run."""
    from ultralytics import YOLO

    weights_path = MODELS_DIR / model_name
    if not weights_path.is_file():
        print(f"初回のみダウンロードします / downloading {model_name} (first run only)...")
        print("  ネットワークに接続できない場合は、次を確認してください:")
        print("  - 社内プロキシの設定 / corporate proxy settings")
        print("  - もう一度このセルを実行する / run this cell again")
        model = YOLO(model_name)          # ダウンロードはカレントフォルダに行われる
        downloaded = Path(model_name)
        if downloaded.is_file():
            shutil.move(str(downloaded), str(weights_path))
            print("保存しました / cached to:", weights_path)
        else:
            raise RuntimeError(
                f"モデルを取得できませんでした / could not obtain {model_name}. "
                "ネットワークを確認して再実行してください / check the network and retry."
            )
    return YOLO(str(weights_path))

---

## Part 1　学習とは何か / What training is

<!-- ne:begin after-heading -->
## Part 1 प्रशिक्षण भनेको के हो
<!-- ne:end -->


### 第1段階：一周と小分け / Stage 1: one pass and small groups

<!-- ne:begin after-heading -->
### पहिलो चरण: एक फेर र साना समूह
<!-- ne:end -->

入力は64枚の学習画像、batch=8です。出力は各batchの画像番号です。
実行前に一周のbatch数を予想してください。epochは全画像を一周することです。

Input is 64 training images with batch=8; output lists the image IDs in each batch.
Predict the batches per pass. One epoch means one pass through all training images.

<!-- ne:begin end -->
इनपुट 64 प्रशिक्षण तस्बिर र batch=8 हो। नतिजा प्रत्येक batch को तस्बिर नम्बर हो। चलाउनुअघि एक फेरमा
कति batch हुन्छन् अनुमान गर्नुहोस्। epoch भनेको सबै तस्बिर एक फेर घुम्नु हो।
<!-- ne:end -->


![64枚を8枚ずつ8組へ / 64 images in eight groups of eight](../../assets/svg/teach_batch_first_pass.svg)

<!-- ne:begin after-image:../../assets/svg/teach_batch_first_pass.svg -->
**ध्यान दिनुपर्ने कुरा**

यो गणना गरिएको योजनाबद्ध चित्र हो। हरेक लाइन एक batch हो र आठ लाइनले एक epoch बन्छ। weight कति पटक
अपडेट हुन्छ भन्ने कुरा यहाँ देखाइएको छैन।

**सोच्नुहोस्**

batch=4 भए एक फेरमा कति batch?
<!-- ne:end -->

**気づいてほしいこと / What to notice**

計算した模式図です。各行が1 batch、8行で1 epochです。重み更新の回数は描いていません。

Calculated schematic: each row is one batch; eight rows make one epoch. Weight-update counts are not shown.

**考えてみてください / Try to answer**

batch=4なら一周に何batch？ / How many batches per pass at batch=4?


In [ ]:
# ▸ この値を変更 / Change this value：指定の値だけを変え、結果を予想します。
# ▸ Change this value: edit only the marked value; predict before running.
image_ids = list(range(64))
simple_batch_size = 8
batches = [image_ids[start:start + simple_batch_size] for start in range(0, len(image_ids), simple_batch_size)]
for batch_index, ids in enumerate(batches, start=1):
    print(f"batch {batch_index}: {ids}")
print("batches per epoch:", len(batches))
print("15 epochs = 15 passes; loader batches:", 15 * len(batches))


8 batchで一周、15 epochなら15周（120回の読み込み）です。重みの更新回数とは別です。
batchを4に変えたときの一周の数を予想します。次の第2段階は更新の詳しい仕組みです。

Eight batches make one pass; 15 epochs make 15 passes (120 loader batches).
This is not the number of weight updates. Predict the batches per pass with batch=4.
Stage 2 below explains why updates may be less frequent.

<!-- ne:begin end -->
8 batch ले एक फेर, 15 epoch भने 15 फेर (डेटा 120 पटक पढाइ) हो। यो weight अपडेट हुने सङ्ख्या होइन।
batch 4 बनाउँदा एक फेरमा कति हुन्छ अनुमान गर्नुहोस्। तलको दोस्रो चरणले अपडेटको विस्तृत क्रम बताउँछ।
<!-- ne:end -->


### 第2段階（補足）：累積と更新 / Stage 2 (extension): accumulation and updates

<!-- ne:begin after-heading -->
### दोस्रो चरण (पूरक): gradient जम्मा र अपडेट
<!-- ne:end -->

![](../../assets/diagrams/teach_epoch_batch.png)

図：8枚ずつ読み込み、名目上8 batch分の勾配をためて更新する例です。
（`batch=8`・`nbs=64`。warmup 中は累積数が変わり得ます。）

Figure: batches contain 8 images; nominal accumulation is 8 batches per update
(`batch=8`, `nbs=64`). Warmup can change that accumulation count.

<!-- ne:begin after-image:../../assets/diagrams/teach_epoch_batch.png -->
चित्र: 8-8 तस्बिर पढेर नाममात्र 8 batch को gradient जम्मा गरेर अपडेट गर्ने उदाहरण (`batch=8`, `nbs=64`)।
warmup को बेला जम्मा हुने सङ्ख्या बदलिन सक्छ।

**ध्यान दिनुपर्ने कुरा**

- **batch** … data loader ले एकै पटक पढ्ने तस्बिर सङ्ख्या। यो तालिममा 8। gradient यहीँ गणना हुन्छ
- **optimizer step (weight अपडेट)** … जम्मा भएको gradient ले weight एक पटक सार्ने। नाममात्र जम्मा 8
  batch हो, तर warmup मा फरक पर्न सक्छ
- **gradient जम्मा** … कति batch को gradient जम्मा गरेपछि एक पटक अपडेट गर्ने (`nbs` ÷ `batch`)
- **epoch** … सबै प्रशिक्षण तस्बिर एक फेर घुम्नु। यो तालिममा 15 फेर

64 ÷ 8 = 8 batch ले एक epoch बन्छ। नाममात्र 8 batch को gradient जम्मा गरेर अपडेट हुन्छ, त्यसैले
**1 batch = 1 अपडेट होइन**। साथै प्रशिक्षणको सुरुमा **warmup** (learning rate र जम्मा गर्ने सङ्ख्या
बिस्तारै लक्ष्यतिर लैजाने अवधि) हुन्छ, त्यसबेला सङ्ख्या 1 बाट क्रमैसँग बढ्छ। त्यसैले "64 ÷ 8 = 8
अपडेट" भनेर गन्न मिल्दैन; अपडेट सङ्ख्या प्रशिक्षक (trainer) ले तय गर्छ।

loss ले "अहिले कति गल्ती भइरहेको छ" जनाउँछ; घट्दै गयो भने प्रशिक्षण अगाडि बढेको हो।

**सोच्नुहोस्**

- 64 प्रशिक्षण तस्बिर र batch 8 हुँदा एक epoch मा data loader कति पटक घुम्छ? तीमध्ये कति weight अपडेट
  बन्छ भन्ने कुरा केले तय गर्छ?
<!-- ne:end -->

**気づいてほしいこと**

- **batch（バッチ）** … データローダが一度にまとめて読み込む枚数。この研修では 8 枚。勾配はここで計算する
- **optimizer step（重みの更新）** … ためた勾配で重みを 1 回動かすこと。名目累積は8 batchですが、warmup 中は変わり得ます
- **勾配の累積（gradient accumulation）** … 何バッチ分の勾配をためてから 1 回更新するか（`nbs` ÷ `batch`）
- **epoch** … 全学習画像を 1 周すること。この研修では 15 周

64 枚 ÷ 8 枚 = 8 batch が 1 epoch です。名目上は勾配を8 batch分ためて更新するため、
**1 batch = 1 回の更新ではありません**。さらに学習の最初には **warmup**（学習率とためる数を
徐々に目標へ近づける期間）があり、その間はためる数が 1 から段階的に増えます。
だから「64 ÷ 8 = 8 回更新」と数えることはできません。更新回数は学習器が決めます。

loss は「今どれだけ間違えているか」で、下がるほど学習が進んでいます。

**What to notice**

- **batch**: how many images the data loader delivers at once — 8 here, and the gradient is computed per batch.
- **optimizer step**: one move of the weights with accumulated gradients; nominally 8 batches, but warmup can change that count.
- **gradient accumulation**: how many batches of gradients are summed before that update (`nbs` / `batch`).
- **epoch**: one pass over the whole training set — 15 here.

64 images / 8 per batch = 8 data-loader batches per epoch. Nominally, 8 batches of gradients are
accumulated, **one batch is not one update**. A **warmup** period also ramps the learning rate and
the accumulated batch count from 1 up to the target, so "64 / 8 = 8 updates" is not a valid count;
the trainer decides it.

The loss says how wrong the model is right now, and it falls as learning progresses.

**考えてみてください / Try to answer**

- 学習画像が 64 枚、batch が 8 枚のとき、1 epoch のデータローダの反復は何回ですか。そのうち
  何回が重みの更新になるかは、何で決まりますか。
- With 64 training images and a batch of 8, how many data-loader iterations are in one epoch, and
  what decides how many of them become weight updates?


#### 学習の繰り返し

<!-- ne:begin after-heading -->
**प्रशिक्षणको दोहोरिने क्रम**
<!-- ne-pair: The training loop -->
<!-- ne:end -->
学習は、次の 4 ステップの **繰り返し** です。

1. **予測する** — 今のモデルで、学習画像を推論する
2. **間違いを測る** — 正解との差を数値にする。これを **損失（loss）** と呼ぶ
3. **少し直す** — 損失が小さくなる方向に、モデルの数値を少し動かす
4. **繰り返す** — 1 周分（全学習画像）を終えると 1 **エポック**

ステップ 3 の「直す」は、毎回起きるとは限りません。勾配を何バッチ分ためてから 1 回更新するかは
設定で決まります（`nbs=64`・`batch=8` の名目累積は8バッチ。warmup 中は変わり得ます）。

損失（loss）が下がっていれば、学習は進んでいます。
ただし **学習用の損失が下がり続けても、実力が上がるとは限りません**（過学習）。

#### The training loop
Training is a **loop** of four steps:

1. **predict** — run the current model on the training images
2. **measure the error** — compare with the truth; the number is the **loss**
3. **adjust a little** — nudge the model's numbers in the direction that reduces the loss
4. **repeat** — one full pass over the training set is one **epoch**

Step 3 does not happen on every iteration. How many batches of gradients are summed before one
update is a setting (`nbs=64`, `batch=8`: nominally 8 batches, variable during warmup).

If the loss falls, learning is progressing. But a falling *training* loss does not guarantee better
real-world skill — that is overfitting.

<!-- ne:begin end -->
प्रशिक्षण चार चरणको **दोहोरिने** काम हो।

1. **अनुमान गर्ने** — अहिलेको model ले प्रशिक्षण तस्बिरमा अनुमान चलाउँछ
2. **गल्ती नाप्ने** — सही उत्तरसँग फरकलाई सङ्ख्यामा बदल्ने। यसलाई **loss** भनिन्छ
3. **अलि सुधार्ने** — loss घट्ने दिशामा model का सङ्ख्या अलि सार्ने
4. **दोहोर्याउने** — सबै प्रशिक्षण तस्बिर एक फेर सकिएपछि एक **epoch** हुन्छ

चरण 3 हरेक पटक हुँदैन। कति batch को gradient जम्मा गरेपछि एक पटक अपडेट गर्ने भन्ने सेटिङले तय गर्छ
(`nbs=64`, `batch=8` को नाममात्र जम्मा 8 batch; warmup मा फरक पर्न सक्छ)।

loss घट्दै छ भने प्रशिक्षण अगाडि बढेको हो। तर **प्रशिक्षणको loss घट्दै जाँदैमा वास्तविक क्षमता बढ्छ
भन्ने छैन** (ओभरफिटिङ)।
<!-- ne:end -->


### 1 つの batch で起きること / what one batch does

<!-- ne:begin after-heading -->
### एउटा batch मा के हुन्छ
<!-- ne:end -->

![](../../assets/diagrams/teach_training_update.png)

図：予測 → 損失 → 勾配 → 更新 の 1 サイクルと、batch・step・epoch の数え方（重みの変化は例示用）。

Figure: one cycle of forward → loss → backward → update, and how batch, step and epoch are counted (weight changes illustrative).

<!-- ne:begin after-image:../../assets/diagrams/teach_training_update.png -->
चित्र: अनुमान → loss → gradient → अपडेट को एक चक्र, र batch, step, epoch गन्ने तरिका (weight को
परिवर्तन उदाहरणका लागि)।

**ध्यान दिनुपर्ने कुरा**

batch (डेटा एकै पटक पढ्ने एकाइ) र weight अपडेट हुने सङ्ख्या फरक हुन्। `nbs=64`, `batch=8` को नाममात्र
जम्मा 8 batch हो। 64 तस्बिरमा data loader 8 पटक घुम्छ, तर warmup मा जम्मा सङ्ख्या पनि बदलिन सक्छ।
वास्तविक अपडेट सधैं एक पटक हो भनेर नठानी चलाइएको सेटिङ र लग जाँच्नुहोस्।

**सोच्नुहोस्**

- 64 तस्बिर र batch=8 मा कति पटक loader घुम्छ, र वास्तविक अपडेट कुन सेटिङले तय गर्छ?
<!-- ne:end -->

**気づいてほしいこと**

バッチ（データをまとめて読む単位）と、重みの更新回数は別です。`nbs=64`・`batch=8` の名目累積は8バッチです。64枚は8回のデータローダ反復になりますが、warmup 中は累積数も変わり得ます。実際の更新回数を常に1回と決めつけず、実行設定とログを確認します。

**What to notice**

A batch and an optimizer update are different units. With `nbs=64` and `batch=8`, nominal accumulation is 8 batches. A pass over 64 images gives 8 loader iterations, but warmup can also change the accumulation count. Do not promise exactly one update; check the run settings and logs.

**考えてみてください / Try to answer**

- 64枚・batch=8 の反復数と、実際の更新回数を決める設定は何ですか。
- For 64 images and batch=8, how many loader iterations occur, and which settings determine actual updates?


#### 用語を一度に整理する

<!-- ne:begin after-heading -->
**शब्दहरू एकै ठाउँमा**
<!-- ne-pair: The terms, all at once -->
<!-- ne:end -->

| 用語 | 意味 | この研修での値 |
|---|---|---|
| エポック（epoch） | 全学習画像を 1 周すること | 15 |
| バッチサイズ（batch） | データローダが一度に読み込む枚数 | 8 |
| 勾配の累積（gradient accumulation） | 何バッチ分の勾配をためてから 1 回更新するか | 名目上8バッチ。warmup 中は変動 |
| warmup | 学習の最初に、学習率とためる数を目標へ近づけていく期間 | 3 epoch 分 |
| 学習率（learning rate） | 1 回の修正の大きさ | 0.01 から徐々に下げる |
| 損失（loss） | 正解との差。小さいほど良い | 学習で下がる |
| 学習（train） | 数値を決める作業 | — |
| 検証（val） | 学習中に実力を確認する | 各エポック後 |

バッチサイズを大きくすると 1 回の処理は重くなります。ただし「大きいほど必ず安定する」とは
言えません。メモリが足りないときは小さくし、安定しているかどうかは **検証の指標** で確かめます。

#### The terms, all at once
| Term | Meaning | Value in this course |
|---|---|---|
| epoch | one full pass over the training set | 15 |
| batch size | images the data loader reads at once | 8 |
| gradient accumulation | how many batches are summed before one update | 8, from `nbs=64` and `batch=8` |
| warmup | a start-up period that ramps the learning rate and the accumulated batch count | 3 epochs |
| learning rate | size of each adjustment | starts at 0.01, then decays |
| loss | distance from the truth; lower is better | falls during training |
| train | the work that determines the numbers | — |
| val | checking progress during training | after each epoch |

A larger batch is heavier per step, but "larger is more stable" is not a rule. Reduce the batch when
you run out of memory, and judge stability on the **validation metrics**.

<!-- ne:begin end -->
| शब्द | अर्थ | यो तालिमको मान |
|---|---|---|
| epoch | सबै प्रशिक्षण तस्बिर एक फेर घुम्नु | 15 |
| batch size | data loader ले एकै पटक पढ्ने तस्बिर सङ्ख्या | 8 |
| gradient जम्मा | कति batch को gradient जम्मा गरेपछि एक पटक अपडेट | नाममात्र 8 batch; warmup मा फरक |
| warmup | प्रशिक्षणको सुरुमा learning rate र जम्मा सङ्ख्या लक्ष्यतिर लैजाने अवधि | 3 epoch जति |
| learning rate | एक पटकको सुधारको आकार | 0.01 बाट बिस्तारै घटाइन्छ |
| loss | सही उत्तरसँगको फरक। सानो भए राम्रो | प्रशिक्षणमा घट्छ |
| प्रशिक्षण (train) | सङ्ख्या तय गर्ने काम | — |
| प्रमाणीकरण (val) | प्रशिक्षणकै बेला क्षमता जाँच्ने | हरेक epoch पछि |

batch size ठुलो बनाउँदा एक पटकको काम भारी हुन्छ। तर "ठुलो भए सधैं स्थिर हुन्छ" भन्न मिल्दैन। मेमोरी
नपुगे सानो बनाउनुहोस् र स्थिर छ कि छैन **प्रमाणीकरणको मापन** ले जाँच्नुहोस्।
<!-- ne:end -->


#### ファインチューニング

<!-- ne:begin after-heading -->
**fine-tuning (थप तालिम)**
<!-- ne-pair: Fine-tuning -->
<!-- ne:end -->
ゼロから学習するのではなく、**COCO で学習済みの `yolo26n.pt` から始めます**。
これを **ファインチューニング** と呼びます。

理由：COCO で学んだ「物の形を見る力」は、塩ビの欠陥にも役立ちます。
少ない画像（64 枚）でも、形の見方を一から覚え直す必要がありません。

とはいえ、**64 枚は非常に少ない** です。必要になる枚数は、欠陥の種類・大きさ・ばらつき・
照明条件で変わります。「何枚あればよい」と一概には言えません。使えるかどうかは、
**代表的な実データ** で確かめます。

#### Fine-tuning
Instead of training from scratch we **start from `yolo26n.pt`, pretrained on COCO**.
This is **fine-tuning**.

Reason: the "shape understanding" learned on COCO transfers usefully to PVC defects, so even with
64 images the model does not have to relearn how to see.

Still, **64 images is very few**. How many you need depends on the defect types, their size, the
variation between parts and the lighting, so no single number applies. Whether the model is usable
is checked on **representative real data**.

<!-- ne:begin end -->
सुरुदेखि प्रशिक्षण नगरी **COCO मा तालिम भएको `yolo26n.pt` बाट सुरु गरिन्छ**। यसलाई **fine-tuning**
भनिन्छ।

कारण: COCO मा सिकेको "वस्तुको आकार हेर्ने क्षमता" PVC को दोषमा पनि काम लाग्छ। थोरै तस्बिर (64) भए पनि
आकार हेर्ने तरिका सुरुदेखि सिक्नु पर्दैन।

तैपनि **64 तस्बिर धेरै थोरै** हो। कति चाहिन्छ भन्ने कुरा दोषको प्रकार, आकार, भिन्नता र उज्यालोको अवस्थाले
तय गर्छ। "कति वटा भए पुग्छ" भनेर एउटै सङ्ख्या भन्न सकिँदैन। काम लाग्छ कि लाग्दैन भन्ने **प्रतिनिधित्व गर्ने
वास्तविक डेटा** मा जाँचिन्छ।
<!-- ne:end -->


---

## Part 2　学習の準備 / Preparing the run

<!-- ne:begin after-heading -->
## Part 2 प्रशिक्षणको तयारी
<!-- ne:end -->


#### `data.yaml` を今の場所に合わせる

<!-- ne:begin after-heading -->
**`data.yaml` लाई अहिलेको स्थानअनुसार मिलाउने**
<!-- ne-pair: Point `data.yaml` at the current location -->
<!-- ne:end -->
`data.yaml` の `path` は、データセット生成時の絶対パスです。
フォルダを移動した場合は、**今の場所に書き直す** 必要があります。

下のセルは、`data.yaml` を **今の研修フォルダに合わせて書き直します**。
他の設定（クラス名など）はそのまま残します。

#### Point `data.yaml` at the current location
`path` in `data.yaml` is the absolute path from generation time. If you moved the
folder, it must be rewritten.

The next cell **rewrites `data.yaml` for the current course folder** while keeping every other
setting.

<!-- ne:begin end -->
`data.yaml` को `path` डेटासेट बनाउँदाको पूरा बाटो हो। फोल्डर सारिएको भए **अहिलेको स्थानमा फेरि लेख्न**
पर्छ। तलको सेलले `data.yaml` **अहिलेको तालिम फोल्डरअनुसार फेरि लेख्छ** र अरू सेटिङ (class नाम आदि)
जस्ताको तस्तै राख्छ।
<!-- ne:end -->


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
dataset_dir = ASSETS / "yolo_dataset"
data_yaml_path = dataset_dir / "data.yaml"

# 既存の設定を読み、path の行だけを今の場所に置き換える
# keep every line, but replace the "path:" line with the current location
lines = data_yaml_path.read_text(encoding="utf-8").splitlines()
updated = []
for line in lines:
    if line.startswith("path:"):
        updated.append(f"path: {dataset_dir.as_posix()}")
    else:
        updated.append(line)
data_yaml_path.write_text("\n".join(updated) + "\n", encoding="utf-8")

print(data_yaml_path.read_text(encoding="utf-8"))
print("data.yaml を更新しました / updated:", data_yaml_path)

#### 学習画像とラベルの対応を最終確認する

<!-- ne:begin after-heading -->
**तस्बिर र लेबलको जोडी अन्तिम पटक जाँच्ने**
<!-- ne-pair: Final check that images and labels pair up -->
<!-- ne:end -->
学習を始める前に、**画像とラベルが 1 対 1 で対応しているか** を確認します。
ここで失敗すると、学習が始まってから変な結果になります。

#### Final check that images and labels pair up
Before training, confirm that **every image has its label**. Failing here produces
strange results only after training has run.

<!-- ne:begin end -->
प्रशिक्षण सुरु गर्नुअघि **तस्बिर र लेबल एक-एक मिलेको छ** जाँच्नुहोस्। यहाँ गल्ती भयो भने प्रशिक्षण
सुरु भएपछि मात्र अनौठो नतिजा देखिन्छ।
<!-- ne:end -->


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
report = []
for split in ("train", "val", "test"):
    images = {p.stem for p in (dataset_dir / "images" / split).glob("*.png")}
    labels = {p.stem for p in (dataset_dir / "labels" / split).glob("*.txt")}
    report.append({
        "split": split,
        "images": len(images),
        "labels": len(labels),
        "missing_labels": len(images - labels),
        "orphan_labels": len(labels - images),
    })

import pandas as pd
print(pd.DataFrame(report).to_string(index=False))

---

## Part 3　学習を実行する / Running the training

<!-- ne:begin after-heading -->
## Part 3 प्रशिक्षण चलाउने
<!-- ne:end -->


#### このセルの学習設定

<!-- ne:begin after-heading -->
**यो सेलको प्रशिक्षण सेटिङ**
<!-- ne-pair: The configuration in this cell -->
<!-- ne:end -->
CPU でも現実的な時間で終わる、**小さな設定** にしています。

| 設定 | 値 | 理由 |
|---|---|---|
| `epochs` | 15 | 少なすぎず、待ち時間も短い |
| `imgsz` | 320 | 元画像が 320 px。CPU でも扱える |
| `batch` | 8 | メモリと速度のバランス |
| `workers` | 0 | Windows のノートブックで安全 |
| `device` | `cpu` | GPU が無くても動く |
| `seed` | 2026 | 再現性のため |

`nbs`（勾配をためる単位、既定 64）と `warmup_epochs`（既定 3.0）は、このセルでは指定していません。
値は保存された `args.yaml` に記録されています。8バッチは名目累積で、warmup 中の更新間隔は異なり得ます。

> **待ち時間はハードウェアで大きく変わります。** 数分で終わることもあれば、
> 15 分以上かかることもあります。**終了時間は保証できません。**
> 遅い場合は `epochs` を 5 に下げてください（学習の流れは同じです）。

#### The configuration in this cell
The configuration is deliberately **small** so it finishes in a reasonable time on CPU.

| Setting | Value | Why |
|---|---|---|
| `epochs` | 15 | enough to see progress, short enough to wait |
| `imgsz` | 320 | the source images are 320 px; CPU friendly |
| `batch` | 8 | balance between memory and speed |
| `workers` | 0 | safe in Windows notebooks |
| `device` | `cpu` | works without a GPU |
| `seed` | 2026 | reproducibility |

`nbs` (the accumulation unit, default 64) and `warmup_epochs` (default 3.0) are not set in this
cell; the saved `args.yaml` records them. Eight batches is nominal accumulation; warmup can
change the actual interval between updates.

**Runtime depends heavily on the hardware**: minutes on some machines, 15+ minutes on others.
**Completion time cannot be guaranteed.** If it is too slow, set `epochs = 5`; the learning process
is the same.

<!-- ne:begin end -->
CPU मा पनि व्यावहारिक समयमा सकिने **सानो सेटिङ** राखिएको छ।

| सेटिङ | मान | कारण |
|---|---|---|
| `epochs` | 15 | धेरै थोरै पनि होइन, पर्खाइ पनि छोटो |
| `imgsz` | 320 | मूल तस्बिर 320 px। CPU मा पनि सम्भव |
| `batch` | 8 | मेमोरी र गतिको सन्तुलन |
| `workers` | 0 | Windows नोटबुकमा सुरक्षित |
| `device` | `cpu` | GPU नभए पनि चल्छ |
| `seed` | 2026 | नतिजा दोहोर्याउन |

`nbs` (gradient जम्मा गर्ने एकाइ, पूर्वनिर्धारित 64) र `warmup_epochs` (पूर्वनिर्धारित 3.0) यो सेलमा
तोकिएका छैनन्; मान सुरक्षित `args.yaml` मा अभिलेख छ। 8 batch नाममात्र जम्मा हो, warmup मा अपडेटको
अन्तर फरक पर्न सक्छ।

> **पर्खाइको समय हार्डवेयरअनुसार धेरै फरक पर्छ।** केही मिनेटमा सकिन पनि सक्छ, 15 मिनेटभन्दा बढी लाग्न
> पनि सक्छ। **सकिने समयको ग्यारेन्टी छैन।** ढिलो भए `epochs` 5 बनाउनुहोस् (प्रशिक्षणको क्रम उही हो)।
<!-- ne:end -->


#### 実行前の確認 / Before starting a job

<!-- ne:begin after-heading -->
#### चलाउनुअघिको जाँच
<!-- ne:end -->

出力は `weights/best.pt`、`results.csv`、`args.yaml` です。所要時間はCPU・メモリ・設定で変わります。
短時間を保証しません。実行しない場合のスキップは維持します。
次の実験は `RUN_NAME` を別名にし、元の実験を残します（同名でも自動で別フォルダを作ります）。
予想：epochを減らした実験の比較には、どの保存ファイルが必要ですか。

Outputs include `weights/best.pt`, `results.csv` and `args.yaml`. Runtime varies by CPU, memory
and settings; there is no short-runtime guarantee. The existing skip path remains.
Use a different `RUN_NAME` for another experiment to preserve the baseline (a reused name also
gets a new output directory). Predict which saved files you need to compare a shorter-epoch run.

<!-- ne:begin end -->
नतिजा `weights/best.pt`, `results.csv` र `args.yaml` हुन्। लाग्ने समय CPU, मेमोरी र सेटिङमा निर्भर छ;
छोटो समयको ग्यारेन्टी छैन। नचलाउने अवस्था (skip) पहिलेजस्तै कायम छ। अर्को प्रयोगका लागि `RUN_NAME`
फरक राख्नुहोस्, पहिलेको प्रयोग जोगाउन (उही नाम राखे पनि स्वतः नयाँ फोल्डर बन्छ)। अनुमान: epoch घटाएको
प्रयोग तुलना गर्न कुन सुरक्षित फाइलहरू चाहिन्छन्?
<!-- ne:end -->


#### 実行スイッチ

<!-- ne:begin after-heading -->
**चलाउने स्विच**
<!-- ne-pair: The run switch -->
<!-- ne:end -->
次のセルを実行すると **学習が始まります**（所要時間は環境で変わります）。
スキップしたい場合は `RUN_TRAINING = False` に書き換えてください。

**長い学習（GPU 向け 100 エポック）は実行しません。** 例として文字列に保存してあります。

#### The run switch
Running the next cell **starts a short training job** (runtime depends on the machine).
Set `RUN_TRAINING = False` to skip it.

**The long GPU configuration (100 epochs) is never executed**; it is stored only as a string example.

<!-- ne:begin end -->
तलको सेल चलाउँदा **प्रशिक्षण सुरु हुन्छ** (लाग्ने समय वातावरणअनुसार फरक पर्छ)। नचलाउन चाहे
`RUN_TRAINING = False` बनाउनुहोस्।

**लामो प्रशिक्षण (GPU का लागि 100 epoch) चलाइँदैन।** उदाहरणको रूपमा string मा मात्र सुरक्षित छ।
<!-- ne:end -->


### コードの読み方 / Reading the code

<!-- ne:begin after-heading -->
### कोड कसरी पढ्ने
<!-- ne:end -->

![](../../assets/svg/teach_epoch_batch_step.svg)

図：バッチ数と更新回数は同じとは限りません。（計算した合成例・模式図。モデルの実測出力ではありません。）

Figure: Batch count need not equal optimizer update count. (computed synthetic / schematic, not measured model output.)

<!-- ne:begin after-image:../../assets/svg/teach_epoch_batch_step.svg -->
चित्र: batch सङ्ख्या र अपडेट सङ्ख्या सधैं उही हुँदैनन्। (गणना गरिएको संश्लेषित उदाहरण र योजनाबद्ध चित्र;
model को वास्तविक नापिएको नतिजा होइन।)

**ध्यान दिनुपर्ने कुरा**

batch सङ्ख्या र अपडेट सङ्ख्या सधैं उही हुँदैनन्। इनपुट र परिवर्तन भएको मान औंलाले पछ्याउनुहोस्।

**सोच्नुहोस्**

64 तस्बिरमा batch=4 बनाउँदा एक epoch मा कति batch हुन्छन्?
<!-- ne:end -->

**気づいてほしいこと**

バッチ数と更新回数は同じとは限りません。 入力と変換後の値を指でたどってください。

**What to notice**

Batch count need not equal optimizer update count. Trace the input and transformed values.

**考えてみてください / Try to answer**

64枚で batch=4 にすると、1 epoch のバッチ数はどうなりますか。

With 64 images and batch=4, how many batches make one epoch?


In [ ]:
# ▸ この値を変更 / Change this value：値を変えて、結果を予想してから実行します。
# ▸ Change this value: edit it below, predict the result, then run.
# ================= 実行スイッチ / run switch =================
RUN_TRAINING = True         # 学習をスキップするときは False にしてください / set to False to skip
RUN_NAME = "pvc_defect_quick"  # ← 別実験は別名 / use a new name for a new experiment
EPOCHS = 15                 # 遅い場合は 5 に下げてください / lower to 5 if it is slow
IMGSZ = 320
BATCH = 8
SEED = 2026
# =============================================================

RUN_LONG_GPU_EXAMPLE = """
# GPU がある場合の例（このノートブックでは実行しません）
# example for a machine with a GPU (not executed here)
model = YOLO("yolo26n.pt")
model.train(
    data=data_yaml_path.as_posix(),
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,          # 0 = first GPU
    workers=4,
    seed=SEED,
    project=str(COURSE_ROOT / "training_results" / "day04"),
    name="pvc_defect_long",
)
"""

if not RUN_TRAINING:
    print("学習は実行していません。/ Training was not started.")
    print("理由 / reason: RUN_TRAINING = False")
    print()
    print("実行するには、このセルの RUN_TRAINING を True に書き換えて再実行してください。")
    print("Set RUN_TRAINING = True in this cell and run it again.")
    print("GPU 向けの長い設定の例は RUN_LONG_GPU_EXAMPLE に入っています（実行はしません）。")
    print()
    print("スキップした場合、以降のセルは動きません。第4日ノートブック03 も実行できません。")
    print("If you skip training, the following cells cannot run, nor can Day 4 Notebook 03.")
else:
    print("学習を開始します / starting training...")
    print(f"epochs={EPOCHS} imgsz={IMGSZ} batch={BATCH} seed={SEED}")

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
# 学習の実行 / run the training
if RUN_TRAINING:
    import time

    train_model = load_pretrained_yolo("yolo26n.pt")
    started = time.time()

    train_results = train_model.train(
        data=data_yaml_path.as_posix(),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        workers=0,                 # Windows のノートブックでは 0 が安全 / safe on Windows
        device="cpu",
        seed=SEED,
        project=str(COURSE_ROOT / "training_results" / "day04"),
        name=RUN_NAME,
        exist_ok=False,
        plots=True,
        verbose=False,
    )

    elapsed_minutes = (time.time() - started) / 60
    run_dir = Path(train_model.trainer.save_dir)
    from training_run import save_training_run
    training_manifest = save_training_run(COURSE_ROOT, run_dir, {
        "epochs": EPOCHS, "imgsz": IMGSZ, "batch": BATCH, "seed": SEED,
        "device": "cpu", "data": str(data_yaml_path.relative_to(COURSE_ROOT)),
    })
    print("評価へ渡す記録 / evaluation handoff:", training_manifest)
    print()
    print(f"学習にかかった時間 / training time: {elapsed_minutes:.1f} minutes")
    print("出力フォルダ / output folder:", run_dir)
else:
    run_dir = None
    print("RUN_TRAINING = False のためスキップしました / skipped because RUN_TRAINING is False")

#### 学習中の画面の読み方

<!-- ne:begin after-heading -->
**प्रशिक्षणका बेला स्क्रिन कसरी पढ्ने**
<!-- ne-pair: How to read the training output -->
<!-- ne:end -->
`verbose=True` にすると、学習中に loss の行が出ます（この研修では静かにするため `False` にしています）。
次の列見出しは、この研修で使う ultralytics 8.4.211 の学習器が表示するものです。

```text
Epoch   GPU_mem  box_loss  cls_loss  l1_loss  Instances  Size
```

| 列 | 意味 |
|---|---|
| `Epoch` | 何周目か |
| `GPU_mem` | 使用メモリ（CPU 実行では 0 に近い値） |
| `box_loss` | 枠の位置の誤差。下がれば位置が合ってきた |
| `cls_loss` | クラス分類の誤差。下がれば種類を当てられるようになった |
| `l1_loss` | 枠の座標の誤差を L1（絶対値の平均）で測った値 |
| `Instances` | そのバッチに含まれたインスタンス（正解）の数 |
| `Size` | 入力画像の一辺の画素（この研修では 320） |

学習のあとに保存される `results.csv` には、同じ内容がエポックごとの列として残ります。
保存した実行（`pvc_defect_quick`）の列は次のとおりです。

```text
epoch,time,train/box_loss,train/cls_loss,train/l1_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/l1_loss,lr/pg0,lr/pg1,lr/pg2
```

`metrics/...` と `val/...` の列は、各エポックのあとに **検証データ** で計算した値です。
学習の損失だけを見ていると気づけない変化は、ここに出ます。

> 古い資料では `dfl_loss`（Distribution Focal Loss）を「境界の細かさの誤差」と説明しています。
> `dfl_loss` は DFL を使う **別のアーキテクチャ** の列で、この研修の YOLO26 には存在しません
> （`results.csv` にもありません）。読み替えず、`l1_loss` を使ってください。

損失が下がり続けていれば学習は進んでいます。ただし **損失だけでは良し悪しを決められません**。
`metrics/mAP50-95` などの検証指標とあわせて読みます。

#### How to read the training output
With `verbose=True` the loss lines appear during training (this course sets `False` to keep the
output quiet). These column headings come from the ultralytics 8.4.211 trainer used here:

```text
Epoch   GPU_mem  box_loss  cls_loss  l1_loss  Instances  Size
```

| Column | Meaning |
|---|---|
| `Epoch` | which pass |
| `GPU_mem` | memory used (close to 0 on CPU) |
| `box_loss` | box position error; falling means boxes fit better |
| `cls_loss` | class error; falling means classes are better |
| `l1_loss` | box-coordinate error measured as L1 (mean absolute error) |
| `Instances` | instances (ground-truth objects) in that batch |
| `Size` | input image side in pixels (320 here) |

`results.csv` keeps the same information as per-epoch columns after training. The recorded run
(`pvc_defect_quick`) has these columns:

```text
epoch,time,train/box_loss,train/cls_loss,train/l1_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/l1_loss,lr/pg0,lr/pg1,lr/pg2
```

The `metrics/...` and `val/...` columns are computed on the **validation** data after each epoch and
show changes that the training loss alone hides.

> Older material describes `dfl_loss` (Distribution Focal Loss) as "boundary refinement error".
> `dfl_loss` belongs to a **different architecture** that uses DFL and does not exist in this
> course's YOLO26 (it is absent from `results.csv`). Do not substitute it; use `l1_loss`.

Falling losses mean learning is progressing, but **the loss alone cannot judge the model**. Read it
together with validation metrics such as `metrics/mAP50-95`.

<!-- ne:begin end -->
`verbose=True` बनाउँदा प्रशिक्षणका बेला loss का लाइन देखिन्छन् (यो तालिममा शान्त राख्न `False` छ)।
तलका स्तम्भ शीर्षक यो तालिममा प्रयोग हुने ultralytics 8.4.211 को trainer ले देखाउने हुन्।

| स्तम्भ | अर्थ |
|---|---|
| `Epoch` | कति औँ फेर |
| `GPU_mem` | प्रयोग भएको मेमोरी (CPU मा 0 नजिक) |
| `box_loss` | आयतको स्थानको त्रुटि। घटे स्थान मिल्दै गएको |
| `cls_loss` | class वर्गीकरणको त्रुटि। घटे प्रकार छुट्याउन सक्दै गएको |
| `l1_loss` | आयतको निर्देशाङ्क त्रुटि L1 (निरपेक्ष मानको औसत) मा नापिएको |
| `Instances` | त्यो batch मा भएका instance (सही लेबल) सङ्ख्या |
| `Size` | इनपुट तस्बिरको एक भुजाको पिक्सेल (यो तालिममा 320) |

प्रशिक्षणपछि सुरक्षित हुने `results.csv` मा उही सामग्री epoch-अनुसारका स्तम्भमा रहन्छ। सुरक्षित
चलाइ (`pvc_defect_quick`) का स्तम्भ यी हुन्:

```text
epoch,time,train/box_loss,train/cls_loss,train/l1_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/l1_loss,lr/pg0,lr/pg1,lr/pg2
```

`metrics/...` र `val/...` स्तम्भ हरेक epoch पछि **प्रमाणीकरण डेटा** मा गणना गरिएका मान हुन्।
प्रशिक्षणको loss मात्र हेर्दा नदेखिने परिवर्तन यहीँ देखिन्छ।

> पुराना सामग्रीले `dfl_loss` (Distribution Focal Loss) लाई "किनाराको सूक्ष्मताको त्रुटि" भनेर व्याख्या
> गर्छ। `dfl_loss` DFL प्रयोग गर्ने **अर्को बनावट** को स्तम्भ हो र यो तालिमको YOLO26 मा छैन
> (`results.csv` मा पनि छैन)। साटेर नपढी `l1_loss` प्रयोग गर्नुहोस्।

loss घट्दै गयो भने प्रशिक्षण अगाडि बढेको हो। तर **loss मात्रले राम्रो-नराम्रो तय गर्न सकिँदैन**;
`metrics/mAP50-95` जस्ता प्रमाणीकरण मापनसँगै पढ्नुहोस्।
<!-- ne:end -->


---

## Part 4　学習結果を保存する / Saving the results

<!-- ne:begin after-heading -->
## Part 4 प्रशिक्षणको नतिजा सुरक्षित गर्ने
<!-- ne:end -->


### 学習 loss と検証 loss を分けて読む / read training and validation loss separately

<!-- ne:begin after-heading -->
### प्रशिक्षण loss र प्रमाणीकरण loss छुट्टाछुट्टै पढ्ने
<!-- ne:end -->

![](../../assets/diagrams/teach_train_val_curves.png)

図：曲線の **形の例** です。あなたの学習結果ではありません。

Figure: an **example of the curve shape** — not your own training run.

<!-- ne:begin after-image:../../assets/diagrams/teach_train_val_curves.png -->
चित्र: वक्रको **आकारको उदाहरण** मात्र हो। तपाईंको प्रशिक्षणको नतिजा होइन।

**ध्यान दिनुपर्ने कुरा**

चित्रको **तेर्सो अक्ष epoch (कति औँ फेर)** र **ठाडो अक्ष loss** हो। दुई लाइन प्रशिक्षण र प्रमाणीकरणका
हुन्, legend ले छुट्याउनुहोस्। प्रशिक्षण loss मात्र हेर्दा epoch बढाउँदै जाँदा राम्रो भएको जस्तो देखिन्छ।
चित्रको डटेड रेखा ("यहाँबाट ओभरफिटिङ") आकार देखाउन बनाइएको उदाहरण हो; वास्तविक डेटामा त्यही ठाउँ
तय हुन्छ भन्ने होइन। **छोटो अवधिको स्थिरता वा वृद्धिले मात्र ओभरफिटिङ भन्न मिल्दैन।**

हेर्नुपर्ने: "प्रमाणीकरण पट्टि घट्न रोकियो कि", "प्रमाणीकरणको मापन (mAP50-95 आदि) अझ बढ्दै छ कि"।
वास्तविक तस्बिरमा देखिएका असफलताको सामग्री पनि सँगै प्रयोग गर्नुहोस्। **प्रशिक्षण loss घट्नु आफैं
"राम्रो भयो" भन्ने प्रमाण होइन।**

**सोच्नुहोस्**

- प्रमाणीकरण loss केही epoch स्थिर रह्यो भने ओभरफिटिङ भन्न मिल्छ? अरू के हेर्नुहुन्छ?
<!-- ne:end -->

**気づいてほしいこと**

図の **横軸は epoch（何周目か）**、**縦軸は loss（損失）** です。2 本の線は学習用と検証用で、
凡例で見分けます。学習 loss だけを見ると、epoch を増やすほど良くなったように見えます。
図の破線（「ここから過学習」）は形を示すための例示で、実際のデータでこの位置が決まるわけでは
ありません。**短い期間の横ばいや上昇だけで過学習と決められるわけでもありません。**

見るのは「検証側が下がり止まったか」「検証の指標（mAP50-95 など）が伸び続けているか」です。
実際の画像で見えた失敗の内容もあわせて使います。
**学習 loss が下がること自体は「良くなった」証拠になりません。**

**What to notice**

The **horizontal axis is the epoch** and the **vertical axis is the loss**. The two lines are the
training and validation curves; use the legend to tell them apart. Training loss alone makes more
epochs look better. The dashed marker ("overfitting starts here") is illustrative — no real dataset
fixes that point — and **a short plateau or a few rising epochs does not prove overfitting** either.

Look at whether the validation curve has stopped falling and whether the validation metrics (mAP50-95
and the others) are still improving, and add the failures you see on real images.
**A falling training loss is not evidence of a better model.**

**考えてみてください / Try to answer**

- 検証 loss が数 epoch 横ばいのとき、過学習だと言い切れますか。他に何を見ますか。
- If validation loss is flat for a few epochs, can you call it overfitting? What else would you
  check?


### 学習ログを 2 本の線で読む / reading the training log as two lines

<!-- ne:begin after-heading -->
### प्रशिक्षण लगलाई दुई लाइनमा पढ्ने
<!-- ne:end -->

![](../../assets/reference_outputs/ref_train_val_curves.png)

図：この研修で実際に行った短い学習（15 epoch, CPU）の記録。左は学習・検証の cls 損失、右は mAP50 と precision。保存された学習ログそのものです。

Figure: the short training run actually performed in this course (15 epochs, CPU). Left: train and val cls loss. Right: mAP50 and precision. This is the saved log.

<!-- ne:begin after-image:../../assets/reference_outputs/ref_train_val_curves.png -->
चित्र: यो तालिममा साँच्चै गरिएको छोटो प्रशिक्षण (15 epoch, CPU) को अभिलेख। बायाँ प्रशिक्षण र प्रमाणीकरणको
cls loss, दायाँ mAP50 र precision। सुरक्षित गरिएको प्रशिक्षण लग नै हो।

**ध्यान दिनुपर्ने कुरा**

एउटा मानभन्दा "दुवै घट्दै छन् कि", "फरक बढ्दै छ कि" भन्ने कुरा पढ्नुहोस्। केही epoch स्थिर रहनु
ओभरफिटिङको प्रमाण होइन। loss का स्तम्भ नाम YOLO26 को वास्तविक नतिजा हुन् (`train/l1_loss`,
`val/l1_loss` छन्; `dfl_loss` छैन)।

**सोच्नुहोस्**

- अन्तिम 3 epoch मा mAP50 बढ्दै छ कि स्थिर छ?
<!-- ne:end -->

**気づいてほしいこと**

読むのは絶対値より「両方が下がり続けるか」「差が開き続けないか」です。数 epoch の横ばいだけで過学習とは言えません。損失の列名は YOLO26 の実際の出力です（`train/l1_loss`・`val/l1_loss` があり、`dfl_loss` はありません）。

**What to notice**

Read whether both curves keep falling and whether the gap keeps widening, not a single value. A few flat epochs are not proof of overfitting. The loss columns are YOLO26's actual outputs (`train/l1_loss`, `val/l1_loss`; there is no `dfl_loss`).

**考えてみてください / Try to answer**

- 最後の 3 epoch で、mAP50 は上がっていますか、横ばいですか。
- In the last three epochs, is mAP50 rising or flat?


#### 4 つを保存する

<!-- ne:begin after-heading -->
**चार कुरा सुरक्षित गर्ने**
<!-- ne-pair: Save four things -->
<!-- ne:end -->
学習が終わると、Ultralytics が出力フォルダに多くのファイルを作ります。
提出物としては、次の 4 つがあれば十分です。

| 種類 | ファイル | 用途 |
|---|---|---|
| 重み | `weights/best.pt` | 推論に使う。**最重要** |
| 設定 | `args.yaml` | どんな設定で学習したかの記録 |
| 指標 | `results.csv` | エポックごとの損失と精度 |
| ログ | `results.csv` から作る要約 | 提出用 |

`best.pt` は **検証データ（val）** で最も良かった時点の重み、`last.pt` は最後の epoch の重みです。
**ふつうは `best.pt` を使います。**

「最も良かった」の基準は、Ultralytics の **fitness** です。この研修で使う ultralytics 8.4.211 の
物体検出では、重みは `[0, 0, 0, 1]`（precision・recall・mAP50・mAP50-95 の順）です。
つまり **fitness = mAP50-95** で、`best.pt` は **検証の mAP50-95 が最も高かった epoch** の重みです。

```text
fitness = 1.0 × mAP50-95        （検証データで計算 / computed on val、ultralytics 8.4.211）
```

`best.pt` を選ぶときも **試験データは見ていません**。だから `best.pt` を選んだあとでも、
試験での評価は 1 回きりにできます。

`last.pt` は最後の epoch の重みです。学習の最終状態を残したいとき、`results.csv` を見て
別の epoch を自分で選びたいとき、`resume` で学習を続けたいときに使います。

#### Save four things
Ultralytics writes many files into the output folder. For submission you need four:

| Kind | File | Use |
|---|---|---|
| weights | `weights/best.pt` | for inference; **the important one** |
| config | `args.yaml` | the record of how training was configured |
| metrics | `results.csv` | loss and accuracy per epoch |
| log | a summary built from `results.csv` | for submission |

`best.pt` is the best checkpoint measured on the **validation** split; `last.pt` is the final
epoch. **Normally you use `best.pt`.**

Ultralytics **fitness** defines "best". In ultralytics 8.4.211 the detection weights are
`[0, 0, 0, 1]` over precision, recall, mAP50 and mAP50-95, so **fitness = mAP50-95**, and `best.pt`
is the epoch with the highest validation mAP50-95.

```text
fitness = 1.0 * mAP50-95        (computed on val, ultralytics 8.4.211)
```

Choosing `best.pt` never looks at the test split, so test can still be evaluated exactly once
afterwards.

`last.pt` is the final epoch. Keep it when you want the end state of the run, when you want to pick
a different epoch yourself from `results.csv`, or when you want to `resume` training.

<!-- ne:begin end -->
प्रशिक्षण सकिएपछि Ultralytics ले नतिजा फोल्डरमा धेरै फाइल बनाउँछ। बुझाउनुपर्ने कामका लागि यी चार
भए पुग्छ।

| प्रकार | फाइल | प्रयोग |
|---|---|---|
| weight | `weights/best.pt` | अनुमानमा प्रयोग। **सबैभन्दा महत्त्वपूर्ण** |
| सेटिङ | `args.yaml` | कुन सेटिङमा प्रशिक्षण गरियो भन्ने अभिलेख |
| मापन | `results.csv` | epoch-अनुसार loss र शुद्धता |
| लग | `results.csv` बाट बनाइएको सारांश | बुझाउनका लागि |

`best.pt` **प्रमाणीकरण डेटा (val)** मा सबैभन्दा राम्रो भएको क्षणको weight हो, `last.pt` अन्तिम epoch
को। **सामान्यतया `best.pt` प्रयोग गरिन्छ।**

"सबैभन्दा राम्रो" को आधार Ultralytics को **fitness** हो। यो तालिममा प्रयोग हुने ultralytics 8.4.211
को वस्तु पहिचानमा weight `[0, 0, 0, 1]` (precision, recall, mAP50, mAP50-95 को क्रम) हुन्छ। अर्थात्
**fitness = mAP50-95** र `best.pt` **प्रमाणीकरणको mAP50-95 सबैभन्दा उच्च भएको epoch** को weight हो।

`best.pt` छान्दा पनि **परीक्षण डेटा हेरिएको छैन**। त्यसैले `best.pt` छानिसकेपछि पनि परीक्षणमा मूल्याङ्कन
एक पटक मात्र गर्न सकिन्छ।

`last.pt` अन्तिम epoch को weight हो। प्रशिक्षणको अन्तिम अवस्था राख्न, `results.csv` हेरेर अर्को epoch
आफैं छान्न, वा `resume` गरेर प्रशिक्षण जारी राख्न प्रयोग हुन्छ।
<!-- ne:end -->


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
if run_dir is not None:
    best_weights = run_dir / "weights" / "best.pt"
    last_weights = run_dir / "weights" / "last.pt"
    args_file = run_dir / "args.yaml"
    results_csv = run_dir / "results.csv"

    for label, path in [("best.pt", best_weights), ("last.pt", last_weights),
                        ("args.yaml", args_file), ("results.csv", results_csv)]:
        exists = path.is_file()
        size = path.stat().st_size if exists else 0
        print(f"{label:<12} exists={exists}  size={size} bytes")

    # 学習結果を提出用フォルダにコピーする / copy the results for submission
    submission_dir = COURSE_ROOT / "submissions" / "day04"
    submission_dir.mkdir(parents=True, exist_ok=True)
    if best_weights.is_file():
        import shutil
        shutil.copyfile(best_weights, submission_dir / "best.pt")
        print()
        print("提出用にコピーしました / copied for submission:", submission_dir / "best.pt")
else:
    print("学習を実行していないため、保存するものはありません。")
    print("Nothing to save because training was not run.")

### 実行結果の例 / Expected output

<!-- ne:begin after-heading -->
### नतिजाको उदाहरण
<!-- ne:end -->

この結果は、この研修で保存した実行の記録です（`yolo26n.pt`、15 epoch、`imgsz=320`、`batch=8`、seed 2026、CPU）。
細かな時間やファイルサイズは環境で変わりますが、4 つのファイルが作られたことを確認してください。

```text
学習を開始します / starting training...
epochs=15 imgsz=320 batch=8 seed=2026

学習にかかった時間 / training time: 2.3 minutes
出力フォルダ / output folder: .../training_results/day04/pvc_defect_quick

best.pt      exists=True  size=5416133 bytes
last.pt      exists=True  size=5416133 bytes
args.yaml    exists=True  size=2091 bytes
results.csv  exists=True  size=2062 bytes
```

These numbers come from the run recorded for this course (`yolo26n.pt`, 15 epochs, `imgsz=320`,
`batch=8`, seed 2026, CPU). Times and file sizes vary between machines; check that the four files
exist.

```text
starting training...
epochs=15 imgsz=320 batch=8 seed=2026

training time: 2.3 minutes
output folder: .../training_results/day04/pvc_defect_quick

best.pt      exists=True
last.pt      exists=True
args.yaml    exists=True
results.csv  exists=True
```

<!-- ne:begin end -->
यो नतिजा यो तालिममा सुरक्षित गरिएको अभिलेख हो (`yolo26n.pt`, 15 epoch, `imgsz=320`, `batch=8`,
seed 2026, CPU)। सूक्ष्म समय र फाइल आकार वातावरणअनुसार फरक पर्छ, तर चार फाइल बनेको कुरा जाँच्नुहोस्।
<!-- ne:end -->


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
if run_dir is not None and results_csv.is_file():
    history = pd.read_csv(results_csv)
    history.columns = [c.strip() for c in history.columns]
    print("列 / columns:", list(history.columns))
    print()
    print(history.tail(5).to_string(index=False))

    # 提出用の要約 / a short summary for the submission
    summary_lines = [
        "# 学習ログの要約 / training log summary",
        "",
        f"epochs={EPOCHS} imgsz={IMGSZ} batch={BATCH} seed={SEED}",
        "",
        history.to_string(index=False),
    ]
    summary_path = COURSE_ROOT / "submissions" / "day04" / "training_log_summary.txt"
    summary_path.write_text("\n".join(summary_lines), encoding="utf-8")
    print()
    print("保存 / saved:", summary_path)
else:
    print("results.csv がありません / results.csv is not available")

#### 学習は成功したと言えるか

<!-- ne:begin after-heading -->
**प्रशिक्षण सफल भयो भन्न मिल्छ?**
<!-- ne-pair: Can we call this training successful? -->
<!-- ne:end -->
損失が下がっただけでは不十分です。**この段階では、まだ性能を確かめていません。**

次のノートブック（04_03）で、
**学習に使っていない画像** で性能を測ります。そこで初めて「使える／使えない」が分かります。

損失が横ばいになったとしても、それだけで過学習とは言えません。判断には
**検証の指標の推移**（`metrics/mAP50-95` など）と、実際の画像で見えた失敗の内容をあわせて
使います。また、実際に使えるかどうかは、**代表的な実データ** で確かめるまで分かりません。

この段階で「精度 95 %」のような数字を出すことは **しません**。
学習用データで測った数字は、実力を表さないからです。

#### Can we call this training successful?
Falling losses are not enough — **performance has not been verified at this stage**.

The next notebook (04_03) measures performance on **images never used for training**. Only then can
you say whether the model is usable.

A loss that stops falling is not by itself overfitting. Judge with the **trend of the validation
metrics** (`metrics/mAP50-95` and the others) and with the failures you see on real images. Whether
the system is usable stays unknown until it is checked on **representative real data**.

We deliberately do **not** quote an accuracy figure here: a number measured on training data does
not reflect real skill.

<!-- ne:begin end -->
loss घट्नु मात्र पुग्दैन। **यो चरणमा क्षमता अझै जाँचिएको छैन।**

अर्को नोटबुक (04_03) मा **प्रशिक्षणमा प्रयोग नभएका तस्बिर** मा क्षमता नापिन्छ। त्यहाँ मात्र
"काम लाग्छ / लाग्दैन" थाहा हुन्छ।

loss स्थिर भयो भने पनि त्यसैले ओभरफिटिङ भन्न मिल्दैन। निर्णयका लागि **प्रमाणीकरण मापनको प्रवृत्ति**
(`metrics/mAP50-95` आदि) र वास्तविक तस्बिरमा देखिएका असफलता सँगै प्रयोग गर्नुहोस्। साथै काम लाग्छ कि
लाग्दैन भन्ने **प्रतिनिधित्व गर्ने वास्तविक डेटा** मा जाँचिएपछि मात्र थाहा हुन्छ।

यो चरणमा "शुद्धता 95%" जस्तो सङ्ख्या **दिइँदैन**; प्रशिक्षण डेटामा नापिएको सङ्ख्याले वास्तविक क्षमता
देखाउँदैन।
<!-- ne:end -->


---

## 演習 / Exercises

<!-- ne:begin after-heading -->
## अभ्यास
<!-- ne:end -->


#### 演習 1（基本）：学習を 5 エポックで実行する

<!-- ne:begin after-heading -->
**अभ्यास 1 (आधारभूत): 5 epoch मा प्रशिक्षण चलाउने**
<!-- ne-pair: Exercise 1 (basic): train for 5 epochs -->
<!-- ne:end -->
`EPOCHS` を 5 にして学習を実行し、`box_loss` の最初と最後の値を記録してください。
あわせて `metrics/mAP50-95` の最初と最後も記録します。損失だけでは判断できないことを確かめます。

| エポック | box_loss | mAP50-95（val） |
|---|---|---|
| 1 | | |
| 5 | | |

- 完了の目安：両方の値を記録し、損失が下がったかどうかを説明できること
- 損失が下がらない、または横ばいでも、それだけで失敗とは言えません

#### Exercise 1 (basic): train for 5 epochs
Set `EPOCHS = 5`, run the training, and record the first and last `box_loss`. Record
`metrics/mAP50-95` for the same epochs — the loss alone is not enough to judge the run.

| Epoch | box_loss | mAP50-95 (val) |
|---|---|---|
| 1 | | |
| 5 | | |

- Done when both values are recorded and you can say whether the loss decreased
- A loss that stops falling, or stays flat, does not by itself mean the run failed

<!-- ne:begin end -->
`EPOCHS` 5 बनाएर प्रशिक्षण चलाउनुहोस् र `box_loss` को पहिलो र अन्तिम मान अभिलेख गर्नुहोस्। साथै
`metrics/mAP50-95` को पहिलो र अन्तिम पनि अभिलेख गर्नुहोस्। loss मात्रले निर्णय गर्न नमिल्ने कुरा
पुष्टि गर्नुहोस्।

| epoch | box_loss | mAP50-95 (val) |
|---|---|---|
| 1 | | |
| 5 | | |

- पूरा भएको मापन: दुवै मान अभिलेख भएका छन् र loss घट्यो कि घटेन भन्न सकिन्छ
- loss नघटे वा स्थिर रहे पनि त्यसैले असफल भन्न मिल्दैन
<!-- ne:end -->


In [ ]:
# ▸ 自分でコードを書く / Write your code：空欄を自分で埋めてから実行します。
# ▸ Write your code: fill in the blanks, then run.
# 演習 1 / Exercise 1
# EPOCHS = 5 にして、上の学習セルを再実行してください / set EPOCHS = 5 and re-run the training cell
# 記録 / record:
first_box_loss = None
last_box_loss = None
print("epoch 1 box_loss:", first_box_loss)
print("epoch 5 box_loss:", last_box_loss)

#### 演習 2（中級）：`imgsz` を変えて比べる

<!-- ne:begin after-heading -->
**अभ्यास 2 (मध्यम): `imgsz` बदलेर तुलना गर्ने**
<!-- ne-pair: Exercise 2 (intermediate): compare `imgsz` -->
<!-- ne:end -->
`IMGSZ` を 160 / 320 で学習し、**学習時間** と **同じ検証指標**（`metrics/mAP50-95`、または
precision・recall）を比べてください。学習 loss の最終値は、入力サイズが変わると数値の意味も
変わるため、精度の比較には使えません。
速さと精度のどちらを取るか、自分の言葉で書いてください。

- 完了の目安：2 回分の時間と検証指標が記録され、トレードオフが説明されていること

#### Exercise 2 (intermediate): compare `imgsz`
Train with `IMGSZ` 160 and 320, then compare **training time** with the **same validation metric**
(`metrics/mAP50-95`, or precision and recall). The final training loss is not an accuracy measure:
its meaning changes with the input size, so do not compare it across sizes. State which trade-off
you would choose and why.

- Done when both runs' times and validation metrics are recorded and the trade-off is explained

<!-- ne:begin end -->
`IMGSZ` 160 / 320 मा प्रशिक्षण गरेर **प्रशिक्षणको समय** र **उही प्रमाणीकरण मापन**
(`metrics/mAP50-95` वा precision, recall) तुलना गर्नुहोस्। प्रशिक्षण loss को अन्तिम मान इनपुट आकार
फेरिँदा अर्थ पनि फेरिने हुनाले शुद्धताको तुलनामा प्रयोग गर्न मिल्दैन। गति र शुद्धता मध्ये कुन रोज्ने
भन्ने कुरा आफ्नै शब्दमा लेख्नुहोस्।

- पूरा भएको मापन: दुई पटकको समय र प्रमाणीकरण मापन अभिलेख भएका छन् र सन्तुलन (trade-off) व्याख्या भएको छ
<!-- ne:end -->


In [ ]:
# ▸ 自分でコードを書く / Write your code：空欄を自分で埋めてから実行します。
# ▸ Write your code: fill in the blanks, then run.
# 演習 2 / Exercise 2
comparison = [
    {"imgsz": 160, "minutes": None, "val_map50_95": None},
    {"imgsz": 320, "minutes": None, "val_map50_95": None},
]
for row in comparison:
    print(row)


#### 演習 3（発展・任意）：学習が進まない原因を考える

<!-- ne:begin after-heading -->
**अभ्यास 3 (वैकल्पिक): प्रशिक्षण किन अगाडि नबढ्ने सोच्ने**
<!-- ne-pair: Exercise 3 (optional challenge): why training might not progress -->
<!-- ne:end -->
損失が下がらない、または NaN になる場合の原因を 4 つ挙げ、
それぞれ「どう確認するか」を書いてください。**必須ではありません。**

完了の目安：原因 4 つと確認方法 4 つが書かれていること。

#### Exercise 3 (optional challenge): why training might not progress
List four reasons why the loss might not decrease (or becomes NaN), and for each one say how you
would check it. This exercise is **not required**.

Done when four causes and four checks are written.

<!-- ne:begin end -->
loss नघट्ने वा NaN हुने चार कारण भन्नुहोस् र प्रत्येकमा "कसरी जाँच्ने" लेख्नुहोस्। **अनिवार्य होइन।**

पूरा भएको मापन: चार कारण र चार जाँच्ने तरिका लेखिएका छन्।
<!-- ne:end -->


In [ ]:
# ▸ 自分でコードを書く / Write your code：空欄を自分で埋めてから実行します。
# ▸ Write your code: fill in the blanks, then run.
# 演習 3 / Exercise 3
causes = [
    ("", ""),
    ("", ""),
    ("", ""),
    ("", ""),
]
for index, (cause, check) in enumerate(causes, start=1):
    print(f"{index}. 原因 / cause: {cause or '-'} -> 確認 / check: {check or '-'}")

---

#### 理解度チェック / Understanding questions

<!-- ne:begin after-heading -->
#### बुझाइ जाँच्ने प्रश्न
<!-- ne:end -->

1. エポック・バッチサイズ・重みの更新（optimizer step）は、それぞれ何が違いますか。
2. 損失（loss）が下がるとは、どういうことですか。それだけで性能を判断できますか。
3. ファインチューニングをする利点を 2 つ挙げてください。
4. `best.pt` は何を基準に選ばれますか。`last.pt` との違いは何ですか。
5. 学習が終わった直後に「精度は 95 % です」と言ってはいけないのはなぜですか。

1. How do an epoch, a batch size and an optimizer step differ?
2. What does it mean that the loss decreases, and is that enough to judge performance?
3. Give two benefits of fine-tuning.
4. What selects `best.pt`, and how does it differ from `last.pt`?
5. Why must you not announce "accuracy is 95%" right after training finishes?
Model answers: `docs/instructor/day04_instructor_guide.md`.

<!-- ne:begin end -->
1. epoch, batch size र optimizer step मा के-के फरक छ?
2. loss घट्नु भनेको के हो? त्यसले मात्र क्षमता तय गर्न मिल्छ?
3. fine-tuning गर्नुका दुई फाइदा भन्नुहोस्।
4. `best.pt` कुन आधारमा छानिन्छ? `last.pt` सँग के फरक छ?
5. प्रशिक्षण सकिएलगत्तै "शुद्धता 95%" भन्न किन मिल्दैन?
<!-- ne:end -->


---

## 困ったときは / Troubleshooting

<!-- ne:begin after-heading -->
## समस्या परेमा
<!-- ne:end -->

エラーが出たら、まず「どのセルで」「どの行で」「どんなメッセージが出たか」を 書き留めてください。提出物にも記録します。

When an error appears, first write down *which cell*, *which line*, and *what the message says*. You will also record it in your submission.

| 症状（日本語） | Symptom (English) | 対処（日本語） | What to do (English) |
| --- | --- | --- | --- |
| 学習が始まらない | Training does not start | `RUN_TRAINING = False` になっています。学習するには `True` にしてください。 | `RUN_TRAINING` is still False. Set it to True and re-run. |
| `RuntimeError: Dataset ... not found` | `RuntimeError: Dataset ... not found` | `data.yaml` の `path` が古い可能性があります。Part 2 のセルで書き直してください。 | `path` in `data.yaml` may be stale. Re-run the Part 2 cell. |
| `CUDA out of memory` / `OutOfMemoryError` | `CUDA out of memory` / `OutOfMemoryError` | `BATCH` を 4、`IMGSZ` を 224 に下げてください。CPU 実行なら起きにくいです。 | Lower `BATCH` to 4 and `IMGSZ` to 224. Rare when running on CPU. |
| 学習が非常に遅い | Training is very slow | CPU 実行では数分〜十数分かかります。`EPOCHS` を 5、`IMGSZ` を 160 にしてください。 | CPU training takes minutes to tens of minutes. Use `EPOCHS=5`, `IMGSZ=160`. |
| `FileNotFoundError: yolo26n.pt` | `FileNotFoundError: yolo26n.pt` | モデルが未取得です。`assets/models/` にファイルがあるか確認してください。 | The model was never obtained. Check `assets/models/`. |
| 損失が下がらない | The loss does not decrease | ラベルの誤り、画像とラベルの不一致、学習率の問題などが考えられます。第4日01 の点検に戻ってください。 | Likely label errors, image/label mismatch, or the learning rate. Repeat the Day 4 Notebook 01 checks. |
| 学習ログに `dfl_loss` が無い | No `dfl_loss` column in the log | `dfl_loss` は DFL を使う別のアーキテクチャの列です。YOLO26 は `l1_loss` を使います。 | `dfl_loss` belongs to a different architecture that uses DFL. YOLO26 logs `l1_loss`. |
| 1 epoch の更新回数が合わない | The update count per epoch looks wrong | 1 batch = 1 回の更新とは限りません。`nbs` と `batch` による勾配の累積と warmup で決まります。 | One batch is not always one update: `nbs`, `batch` accumulation and warmup decide it. |

<!-- ne:begin end -->
त्रुटि आएपछि पहिले "कुन सेलमा", "कुन लाइनमा", "के सन्देश आयो" भनेर लेख्नुहोस्। बुझाउनुपर्ने काममा पनि
त्यही अभिलेख राख्नुहोस्।

| लक्षण | के गर्ने |
| --- | --- |
| प्रशिक्षण सुरु हुँदैन | `RUN_TRAINING = False` छ। प्रशिक्षण गर्न `True` बनाउनुहोस्। |
| `RuntimeError: Dataset ... not found` | `data.yaml` को `path` पुरानो हुन सक्छ। Part 2 को सेलले फेरि लेख्नुहोस्। |
| `CUDA out of memory` / `OutOfMemoryError` | `BATCH` 4 र `IMGSZ` 224 बनाउनुहोस्। CPU मा यो कम हुन्छ। |
| प्रशिक्षण धेरै ढिलो | CPU मा केही मिनेटदेखि दर्जनौं मिनेट लाग्छ। `EPOCHS` 5 र `IMGSZ` 160 बनाउनुहोस्। |
| `FileNotFoundError: yolo26n.pt` | model लिइएको छैन। `assets/models/` मा फाइल छ कि जाँच्नुहोस्। |
| loss घट्दैन | लेबलको गल्ती, तस्बिर र लेबल नमिल्नु, learning rate को समस्या हुन सक्छ। दिन ४ नोटबुक ०१ को जाँचमा फर्कनुहोस्। |
| प्रशिक्षण लगमा `dfl_loss` छैन | `dfl_loss` DFL प्रयोग गर्ने अर्को बनावटको स्तम्भ हो। YOLO26 ले `l1_loss` लेख्छ। |
| एक epoch को अपडेट सङ्ख्या मिलेन | 1 batch = 1 अपडेट नहुन सक्छ। `nbs` र `batch` को gradient जम्मा तथा warmup ले तय गर्छ। |
<!-- ne:end -->


## 完了チェックリスト / Completion checklist

<!-- ne:begin after-heading -->
## सम्पन्नता जाँचसूची
<!-- ne:end -->

**必須の項目だけ** を並べています。**演習 3（発展・任意）は必須ではありません**。未実施でも完了です。
時間が余ったときだけ取り組んでください。

Only the **required** items are listed. **Exercise 3 (optional challenge) is not required**; leaving
it undone still counts as complete. Do it only if you have spare time.

すべてにチェックが付いたら、このノートブックは完了です。

This notebook is complete when every box is checked.

- [ ] Part 1〜Part 4 を読み、すべてのコードセルを実行した
      - [ ] Read Parts 1-4 and ran every code cell
- [ ] `data.yaml` を今の場所に合わせて書き直した
      - [ ] Rewrote `data.yaml` for the current location
- [ ] 画像とラベルの対応を確認した
      - [ ] Confirmed images and labels pair up
- [ ] 学習を実行した（`RUN_TRAINING = True`）
      - [ ] Ran the training (`RUN_TRAINING = True`)
- [ ] `best.pt` と `results.csv` が生成された
      - [ ] `best.pt` and `results.csv` exist
- [ ] 学習ログの要約を保存した
      - [ ] Saved the training summary
- [ ] 演習 1〜2（必須）を実施した
      - [ ] Completed exercises 1-2 (required)
- [ ] 理解度チェックの 5 問に答えた
      - [ ] Answered the 5 questions
- [ ] ノートブックを保存した
      - [ ] Saved the notebook

### 任意の項目 / Optional items

- [ ] 演習 3（発展・任意）：学習が進まない原因を 4 つ挙げた
      - [ ] Exercise 3 (optional challenge): listed four possible causes

<!-- ne:begin end -->
**अनिवार्य बुँदा मात्र** दिइएका छन्। **अभ्यास 3 (वैकल्पिक) अनिवार्य होइन**; नगरे पनि पूरा मानिन्छ।
समय बाँकी भएमा मात्र गर्नुहोस्। सबैमा चिन्ह लागेपछि यो नोटबुक पूरा हुन्छ।

**अनिवार्य**

- [ ] Part 1–Part 4 पढेर सबै code सेल चलाएँ
- [ ] `data.yaml` अहिलेको स्थानअनुसार फेरि लेखें
- [ ] तस्बिर र लेबलको जोडी जाँचें
- [ ] प्रशिक्षण चलाएँ (`RUN_TRAINING = True`)
- [ ] `best.pt` र `results.csv` बने
- [ ] प्रशिक्षण लगको सारांश सुरक्षित गरें
- [ ] अभ्यास 1–2 (अनिवार्य) गरें
- [ ] बुझाइ जाँच्ने 5 प्रश्नको उत्तर दिएँ
- [ ] नोटबुक सुरक्षित गरें

**वैकल्पिक**

- [ ] अभ्यास 3 (वैकल्पिक): प्रशिक्षण नबढ्ने चार कारण भनें
<!-- ne:end -->


## さらに学ぶ / Further learning（任意 / optional）

### 参考リンク / Reference links

<!-- ne:begin after-heading -->
## थप अध्ययन (वैकल्पिक)
<!-- ne:end -->

- [Ultralytics Train モード](https://docs.ultralytics.com/modes/train/)
- [Ultralytics ハイパーパラメータ](https://docs.ultralytics.com/guides/hyperparameter-tuning/)
- [PyTorch 学習の基礎](https://pytorch.org/tutorials/beginner/basics/optimization_tutorial.html)

### 追加チャレンジ / Extra challenge（必須ではありません / not mandatory）

`BATCH` を 4 と 16 で学習し、損失の下がり方の「なめらかさ」を比べてください。

Train with `BATCH` 4 and 16 and compare how smoothly the loss decreases.

<!-- ne:begin end -->
### सन्दर्भ लिङ्क

- [Ultralytics Train मोड](https://docs.ultralytics.com/modes/train/)
- [Ultralytics hyperparameter](https://docs.ultralytics.com/guides/hyperparameter-tuning/)
- [PyTorch प्रशिक्षणको आधार](https://pytorch.org/tutorials/beginner/basics/optimization_tutorial.html)

### थप चुनौती (अनिवार्य होइन)

`BATCH` 4 र 16 मा प्रशिक्षण गरेर loss घट्ने "सहजता" (smoothness) तुलना गर्नुहोस्।
<!-- ne:end -->
